# Isotropic power spectrum

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/L16B06")

## Load test dataset

In [ ]:
import xarray as xr

## Open a netCDF file in a xarray dataset
fname = 'data/garachico2048.ens.nc'
ds    = xr.open_dataset(fname)
test  = ds['tephra_col_mass']

## Load a pre-trained VAE

In [ ]:
import torch
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale

## Load weight parameters and some metadata
fname = output_dir / 'model.pt'
checkpoint = torch.load(fname)

## Recreate the model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
model.load_state_dict(checkpoint['model_state_dict'])

## Generate a VAE ensemble

In [ ]:
## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
transform = MinMaxScale(min_value, max_value)

## Generate nens new samples
nens = 5000
z = torch.randn(nens, checkpoint['LATENT_DIM'])
with torch.no_grad():
    new_sample = model.decode(z)
    x = transform.invert(new_sample).squeeze()

## Compute spectrum

In [ ]:
from modules.metrics import isotropic_spectrum_ensemble

## Data map
data_map = {
    'test': isotropic_spectrum_ensemble(test.values),
    'vae':  isotropic_spectrum_ensemble(x.numpy())
}

## Plot configuration

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 200

DEFAULTS = dict(
    label = None,
    color = 'r',
    ls = 'None',
    alpha = 1.0,
    marker = None,
)

conf_all = {
    'test': dict(label='Test dataset', ls='solid'),
    'vae':  dict(label='VAE - Kernel 3x3', alpha=0.5, color = 'g', marker='^'),
    'vae5': dict(label='VAE - Kernel 5x5', alpha=0.4, color = 'b', marker='+'),   
}

## Plot IPS

In [ ]:
fig, ax = plt.subplots()

ax.plot([0.333,0.333],[0,1E-6], 'k--', alpha=0.4)
for key, value in data_map.items():
    conf = DEFAULTS | conf_all[key]
    k, Pk, _ = value
    ax.plot(k,Pk,
            color  = conf['color'],
            label  = conf['label'],
            alpha  = conf['alpha'],
            marker = conf['marker'],
            ls     = conf['ls'],
           )

ax.set(xlabel=r'Wavenumber, k [$1/\Delta x$]',
       ylabel=r'Isotropic power spectrum, P(k) [$g^2/m^4$]',
       yscale='log')

ax.annotate(r'$\sim 3\Delta x$'+'\nThree grid cells', xy=(0.333, 1E-6), xycoords='data',
            xytext=(0.25, .4), textcoords='axes fraction',
            va='top', ha='center',
            fontsize = 14,
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
ax.legend()